In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

In [ ]:
dataset, dataset_info = tfds.load("malaria", with_info = True, split = "train", as_supervised = True)

In [ ]:
def split(dataset, train_size, val_size):
    dataset_size = len(dataset)
    train_dataset = dataset.take(int(dataset_size*train_size))
    val_test_dataset = dataset.skip(int(dataset_size*train_size))
    val_dataset = val_test_dataset.take(int(dataset_size*val_size))
    test_dataset = val_test_dataset.skip(int(dataset_size*val_size))
    return train_dataset, val_dataset, test_dataset

In [ ]:
train_dataset, val_dataset, test_dataset = split(dataset, 0.8, 0.1)

In [ ]:
resize_layer = tf.keras.layers.Resizing(256, 256)
rescale_layer = tf.keras.layers.Rescaling(1/255)

def resize_rescale(image, label):
    image = resize_layer(image)
    image = rescale_layer(image)
    return image, label

train_dataset = train_dataset.map(resize_rescale)
val_dataset = val_dataset.map(resize_rescale)

test_dataset = test_dataset.map(resize_rescale)

In [ ]:
import albumentations as A
transforms = A.Compose([A.CLAHE(clip_limit=(1, 8), tile_grid_size=(16, 16), p=1.0)])

def aug_fn(image):
  image = tf.image.convert_image_dtype(image, tf.uint8)
  image = image.numpy()
  dictionary = {"image": image}
  transformed_dictionary = transforms(**dictionary)
  image = transformed_dictionary["image"]
  image = tf.image.convert_image_dtype(image, tf.float32)
  image = tf.image.resize(image, [256, 256])
  return image

def process_data(image, label):
  transformed_image = tf.numpy_function(func = aug_fn, inp = [image], Tout = tf.float32)
  transformed_image = tf.reshape(transformed_image, (256, 256, 3))
  return transformed_image, label

train_dataset = train_dataset.map(process_data)
val_dataset = val_dataset.map(process_data)

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 1.4.18 (you have 1.4.15). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [ ]:
train_dataset = train_dataset.shuffle(buffer_size = 162).batch(32)
val_dataset = val_dataset.shuffle(buffer_size = 162).batch(32)
test_dataset = test_dataset.batch(1)

In [ ]:
from keras.layers import Normalization, Dense, Input, Conv2D, MaxPool2D, Flatten, BatchNormalization, RandomRotation, RandomFlip

augment_layers = tf.keras.Sequential([RandomFlip(mode='horizontal_and_vertical'), RandomRotation(0.25)])

model = tf.keras.Sequential([
    Input(shape = (256, 256, 3)),

    Conv2D(filters = 4, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 16, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 64, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 64, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Conv2D(filters = 32, kernel_size = 3, strides = 1, padding = 'valid', activation = "relu", use_bias = False),
    BatchNormalization(),
    MaxPool2D(pool_size = 2, strides = 2),

    Flatten(),

    Dense(500, activation = "relu"),
    BatchNormalization(),

    Dense(250, activation = "relu"),
    BatchNormalization(),

    Dense(1, activation = "sigmoid")
])

In [ ]:
from keras.losses import BinaryCrossentropy

def custom_bce(factor):
    bce = BinaryCrossentropy()
    def loss(y_true, y_pred):
        return bce(y_true, y_pred) * factor
    return loss

In [ ]:
class CustomBce(tf.keras.losses.Loss):
  def __init__(self, factor):
    super(CustomBce, self).__init__()
    self.factor = factor
  def call(self, y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()
    return bce(y_true, y_pred) * self.factor

In [ ]:
from keras.metrics import BinaryAccuracy

def custom_accuracy(factor):
    acc = BinaryAccuracy()
    def metric(y_true, y_pred):
        return acc(y_true, y_pred) * factor
    return metric

In [ ]:
class CustomAccuracy(tf.keras.metrics.Metric):
  def __init__(self, factor, name = 'custom accuracy'):
    super(CustomAccuracy, self).__init__()
    self.factor = factor
    self.accuracy = self.add_weight(name = name, initializer = 'zeros')

  def update_state(self, y_true, y_pred, sample_weight=None):
    self.accuracy.assign(BinaryAccuracy(y_true, y_pred) * self.factor)

  def result(self):
    return self.accuracy

  def reset_state(self):
    self.accuracy.assign(0)

In [ ]:
from keras.optimizers import Adam
from keras.metrics import TruePositives, TrueNegatives, FalsePositives, FalseNegatives, Precision, Recall, AUC
metrics = [custom_accuracy(1), TruePositives(), TrueNegatives(), FalsePositives(), FalseNegatives(), Precision(), Recall(), CustomAccuracy(1)]
model.compile(optimizer = Adam(), loss = CustomBce(1), metrics = metrics)

In [ ]:
history = model.fit(train_dataset, validation_data = val_dataset, epochs = 5, verbose = 1)

Epoch 1/5


KeyboardInterrupt: 

In [ ]:
from keras.losses import BinaryCrossentropy

def custom_bce(y_true, y_pred):
  bce = BinaryCrossentropy()
  loss = bce(y_true, y_pred)
  return loss

In [ ]:
from keras.metrics import BinaryAccuracy
from keras.optimizers import Adam

metric = BinaryAccuracy()
optimizer = Adam(learning_rate = 0.01)


In [ ]:
@tf.function
def training_block(x_batch, y_batch):
  with tf.GradientTape() as recorder:
      y_pred = model(x_batch, training = True)
      loss = custom_bce(y_batch, y_pred)

  partial_derivatives = recorder.gradient(loss, model.trainable_weights)
  optimizer.apply_gradients(zip(partial_derivatives, model.trainable_weights))
  metric.update_state(y_batch, y_pred)
  return loss

@tf.function
def val_block(x_batch_val, y_batch_val):
    y_pred_val = model(x_batch_val, training = False)
    loss_val = custom_bce(y_batch_val, y_pred_val)
    metric.update_state(y_batch_val, y_pred_val)
    return loss_val

In [ ]:
def neuralearn(model, metric, optimizer, train_dataset, val_dataset, epochs):
  for epoch in range(epochs):
    for step, (x_batch, y_batch) in enumerate(train_dataset):
      loss = training_block(x_batch, y_batch)

    print("Training Loss", loss)
    print("The accuracy is: ", metric.result())
    metric.reset_states()

    for (x_batch_val, y_batch_val) in val_dataset:
      loss_val = val_block(x_batch_val, y_batch_val)

    print("The Validation loss", loss_val)
    print("The Validation accuracy is: ", metric.result())
    metric.reset_states()

In [ ]:
neuralearn(model, metric, optimizer, train_dataset, val_dataset, 2)

KeyboardInterrupt: 